# Env38 - Geometry, Circuity, and Orientation Metrics

Professor-facing inspection of the accepted planar Env38 geometry baseline through the tested production module.

## 1. Purpose and scope
Reproduce and explain the accepted geometry and orientation results without modifying the canonical network or published analysis.

## 2. Dynamic accepted-run selection
The canonical and analysis `latest.json` pointers are resolved and cross-checked at runtime; no active run identifier is embedded here.

In [ ]:
from pathlib import Path
import csv, json, math, sys

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / 'environment.yml').is_file() and (candidate / 'config' / 'environments.yaml').is_file():
            return candidate
    raise RuntimeError('Repository root not found')
REPOSITORY = find_repository(Path.cwd().resolve())
if str(REPOSITORY / 'src') not in sys.path:
    sys.path.insert(0, str(REPOSITORY / 'src'))
from geogami_morphology.environments import get_environment
from geogami_morphology.metrics_geometry import analyze_geometry, load_geometry_config
ENVIRONMENT, CANONICAL_RUN = 'env38', 'latest'
environment = get_environment(ENVIRONMENT, repository_root=REPOSITORY)
canonical_pointer = json.loads(environment.canonical_latest.read_text(encoding='utf-8'))
analysis_pointer = json.loads((environment.analysis_root / 'latest.json').read_text(encoding='utf-8'))
assert canonical_pointer['run_id'] == analysis_pointer['canonical_run_id']
assert (REPOSITORY / canonical_pointer['canonical_path']).resolve() == (REPOSITORY / analysis_pointer['canonical_path']).resolve()
print('ENV38_GEOMETRY_POINTER_RESOLUTION: PASS')

## 3. Versioned configuration and local Cartesian CRS
The versioned configuration fixes tolerances, population standard deviation (`ddof=0`), orientation convention, bins, weighting, entropy base, and symmetry. GeoGami Local Cartesian coordinates are not longitude/latitude. Distances are reported in local units; do not call them metres unless a justified scale transformation is established. This notebook invokes no geographic or projection methods.

In [ ]:
config = load_geometry_config(project_root=REPOSITORY)
result = analyze_geometry(ENVIRONMENT, canonical_run=CANONICAL_RUN, latest_path=environment.canonical_latest, project_root=REPOSITORY, publish=False)
assert result.selection.run_id == canonical_pointer['run_id']
assert len(result.canonical.edges) == 69
print(f'GEOMETRY_CONFIG: {config.path.relative_to(REPOSITORY)} | bins={config.orientation_bins} | width={config.orientation_bin_width_degrees} degrees')
print('ENV38_GEOMETRY_CALCULATION: PASS')

## 4. LineString length and chord length
For edge *i*, geometry length `L_i` is the sum of its LineString segment lengths. Chord length `D_i` is the Euclidean distance between the first and last vertices.

## 5. Edge and network circuity
Eligible edge circuity is `L_i / D_i`. Network circuity is `sum(L_i) / sum(D_i)`, which is not the arithmetic mean of edge ratios. Excess length is `L_i - D_i`.

## 6. Axial orientation convention
Orientation is axial over `[0, 180)` degrees: reversing a line does not change its orientation. Zero degrees points north, 90 degrees points east, and angles increase clockwise.

## 7. Histogram discretization
The accepted histogram uses 36 bins across 180 degrees, hence 5 degrees per bin, with the configured boundary tolerance.

## 8. Chord and segment populations
Chord orientation gives one equally weighted observation per eligible physical edge. Segment orientation retains every nonzero LineString segment and weights each observation by its local segment length.

## 9. Shannon orientation entropy
For bin probabilities `p_j`, Shannon entropy is `H = -sum(p_j ln(p_j))` using the natural logarithm; zero-probability bins contribute zero. Normalized entropy is `H / ln(36)`.

## 10. Fourfold orientation order
The accepted fourfold order parameter is the magnitude of the weighted complex mean `|sum(w_k exp(i 4 theta_k)) / sum(w_k)|`, evaluated separately for chords and segments.

## 11. Geometry and orientation result tables
The calculated production rows are checked against the accepted JSON summaries below.

In [ ]:
def metrics_from_json(path):
    value = json.loads(path.read_text(encoding='utf-8'))
    rows = value['metrics'] if isinstance(value, dict) and 'metrics' in value else value
    return {row['metric_name']: row['value'] for row in rows}
geometry_stored = metrics_from_json(result.output_dir / 'geometry_metrics_summary.json')
orientation_stored = metrics_from_json(result.output_dir / 'orientation_metrics_summary.json')
geometry_calculated = {row['metric_name']: row['value'] for row in result.calculation.geometry_summary_rows}
orientation_calculated = {row['metric_name']: row['value'] for row in result.calculation.orientation_summary_rows}
assert geometry_calculated == geometry_stored
assert orientation_calculated == orientation_stored
for name in ['total_network_length_local','mean_edge_length_local','median_edge_length_local','total_chord_length_local','mean_edge_circuity','network_circuity','geometry_segment_count','zero_chord_edge_count']:
    print(f'{name}: {geometry_calculated[name]}')
for name in ['chord_orientation_entropy','chord_orientation_normalized_entropy','segment_length_weighted_orientation_entropy','segment_length_weighted_normalized_entropy','phi_chord','phi_segment_length_weighted']:
    print(f'{name}: {orientation_calculated[name]}')
print('ENV38_GEOMETRY_STORED_CROSSCHECK: PASS')

## 12. Absolute excess-length audit
The accepted edge table permits a transparent aggregate audit: `sum(L_i - D_i)` over the 69 physical edges. This reads published row values and does not introduce a competing geometry algorithm.

In [ ]:
with (result.output_dir / 'edge_geometry_metrics.csv').open(newline='', encoding='utf-8') as stream:
    edge_rows = list(csv.DictReader(stream))
absolute_excess = math.fsum(float(row['excess_length_local']) for row in edge_rows)
assert len(edge_rows) == 69
print(f'absolute_excess_length_local: {absolute_excess}')
print('ENV38_ABSOLUTE_EXCESS_AUDIT: PASS')

## 13. Accepted figure inventory
Five existing SVGs are reused: edge length, edge circuity, chord orientation histogram, chord orientation rose, and segment-length-weighted orientation histogram. Notebook execution does not regenerate them.

In [ ]:
figure_names = ['edge_length_distribution.svg','edge_circuity_distribution.svg','chord_orientation_histogram.svg','chord_orientation_rose.svg','segment_length_weighted_orientation_histogram.svg']
figure_paths = [result.output_dir / 'figures' / name for name in figure_names]
assert all(path.is_file() for path in figure_paths)
print('ENV38_GEOMETRY_FIGURES: ' + ', '.join(figure_names))

## 14. Artifact inventory and provenance
Accepted summaries, edge and segment tables, methodology, figures, and manifest remain bound to the canonical hashes, metric configuration, Git provenance, and software environment.

## 15. Limitations and interpretation
These are planar descriptors of one accepted realization. Entropy depends on binning and population; order depends on fourfold symmetry; segment results depend on stored vertex segmentation. Values are descriptive and are not evidence that one environment is better or causes behavioral outcomes.

## 16. Reproducibility summary
The notebook resolves pointers dynamically, invokes the tested production module with `publish=False`, cross-checks accepted machine-readable artifacts, and performs no network access or source mutation.

In [ ]:
print(f'ACCEPTED_ENV38_RUN: {result.selection.run_id}')
print(f'GEOMETRY_METRICS: {len(result.calculation.geometry_summary_rows)} | ORIENTATION_METRICS: {len(result.calculation.orientation_summary_rows)}')
print('NOTEBOOK_07_EXECUTION: PASS')